In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("weatherAUS.csv")
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [3]:
df.shape

(145460, 23)

## Step 1 — Drop rows where the target `RainTomorrow` is null

These rows have no label, so they can't be used for training or evaluation. We drop them up front (this also removes most rows where `RainToday`/`Rainfall` are missing, since they overlap heavily).

In [4]:
# Step 1 — Drop rows where the target RainTomorrow is null
before = len(df)
df = df.dropna(subset=['RainTomorrow']).reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows with null RainTomorrow; {len(df):,} rows remain")
print(df['RainTomorrow'].value_counts(dropna=False))

Dropped 3,267 rows with null RainTomorrow; 142,193 rows remain
RainTomorrow
No     110316
Yes     31877
Name: count, dtype: int64


## Step 2 — Chronological train / validation / test split

We split **before** any imputation or scaling so that statistics learned later (medians, means, scaler) come only from the training set — no leakage from the future.

- **Train:** 2008–2014
- **Validation:** 2015
- **Test:** 2016 and later

In [5]:
# Step 2 — Chronological train / validation / test split
# Parse Date and split by year so no future information leaks into the past.
df['Date'] = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

df_train = df[df['Year'] <= 2014].copy()   # 2008-2014  -> training
df_val   = df[df['Year'] == 2015].copy()   # 2015       -> validation
df_test  = df[df['Year'] >= 2016].copy()   # 2016+      -> test

for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    yrs = f"{part['Year'].min()}-{part['Year'].max()}"
    pos = (part['RainTomorrow'] == 'Yes').mean()
    print(f"{name:5s}: {len(part):>7,} rows  years {yrs}  RainTomorrow=Yes {pos:.1%}")

train:  98,988 rows  years 2007-2014  RainTomorrow=Yes 22.5%
val  :  17,231 rows  years 2015-2015  RainTomorrow=Yes 21.2%
test :  25,974 rows  years 2016-2017  RainTomorrow=Yes 22.9%


## Step 3 — Wind direction cyclical (sin/cos) encoding

Wind directions are a 16-point compass (N, NNE, ... NNW). Treating them as unordered categories loses the fact that N and NNW are adjacent. We map each direction to its compass bearing in degrees, then to `sin` and `cos`, giving 2 numeric features per column that wrap around correctly. Done for all three: `WindGustDir`, `WindDir9am`, `WindDir3pm`.

In [6]:
# Step 3 — Wind direction cyclical (sin/cos) encoding for all 3 direction columns
# 16-point compass -> degrees -> radians -> (sin, cos).
# sin/cos keeps adjacent directions close (e.g. N and NNW) and is a deterministic
# transform (no fitting), so there is no train/test leakage in applying it per split.

COMPASS = {
    'N': 0.0,   'NNE': 22.5,  'NE': 45.0,   'ENE': 67.5,
    'E': 90.0,  'ESE': 112.5, 'SE': 135.0,  'SSE': 157.5,
    'S': 180.0, 'SSW': 202.5, 'SW': 225.0,  'WSW': 247.5,
    'W': 270.0, 'WNW': 292.5, 'NW': 315.0,  'NNW': 337.5,
}

WIND_DIR_COLS = ['WindGustDir', 'WindDir9am', 'WindDir3pm']

def encode_wind_dirs(frame, cols=WIND_DIR_COLS):
    """Return a copy of `frame` with *_sin / *_cos columns added for each wind-direction column.
    NaN directions stay NaN in the sin/cos columns and are handled later during imputation."""
    out = frame.copy()
    for c in cols:
        rad = np.deg2rad(out[c].map(COMPASS).astype('float64'))
        out[c + '_sin'] = np.sin(rad)
        out[c + '_cos'] = np.cos(rad)
    return out

df_train = encode_wind_dirs(df_train)
df_val   = encode_wind_dirs(df_val)
df_test  = encode_wind_dirs(df_test)

# Sanity check: show the original + encoded columns for one wind field
df_train[['WindGustDir', 'WindGustDir_sin', 'WindGustDir_cos']].head()

,WindGustDir,WindGustDir_sin,WindGustDir_cos
0,W,-1.000000,-1.836970e-16
1,WNW,-0.923880,3.826834e-01
2,WSW,-0.923880,-3.826834e-01
3,NE,0.707107,7.071068e-01
4,W,-1.000000,-1.836970e-16


## Step 4 — Encode binary Yes/No columns

`RainToday` (a feature) and `RainTomorrow` (the target) are `Yes`/`No` strings. We map them to `1`/`0`. The target has no nulls left (dropped in Step 1); `RainToday` still has a few nulls, which stay `NaN` for now and get handled in the imputation step.

In [7]:
# Step 4 — Encode binary Yes/No columns to 1/0
yn_map = {'Yes': 1, 'No': 0}

for part in (df_train, df_val, df_test):
    part['RainToday']    = part['RainToday'].map(yn_map)
    part['RainTomorrow'] = part['RainTomorrow'].map(yn_map)

# Check: target is fully 0/1 (no NaN), RainToday may still have a few NaN to impute later
print("RainTomorrow (train) value counts:")
print(df_train['RainTomorrow'].value_counts(dropna=False))
print("\nRainToday (train) null count:", df_train['RainToday'].isna().sum())

RainTomorrow (train) value counts:
RainTomorrow
0    76705
1    22283
Name: count, dtype: int64

RainToday (train) null count: 1000


## Step 5 — Impute temperature and rainfall columns

Fill missing temperature and rainfall values using a group statistic learned from the **training set only**, with a fallback chain so every null gets filled:

**`(Location, Month)` → `Location` → global**

- **Temperature** (`MinTemp`, `MaxTemp`, `Temp9am`, `Temp3pm`) → **mean** (roughly symmetric).
- **Rainfall** → **median** (right-skewed; mean is distorted by storm days).

No missing-indicator flags here — these columns are ≤2.5% missing, so a flag would be almost all zeros. Flags are reserved for the high-missing columns (Sunshine / Evaporation / Cloud) in a later step.

In [8]:
# Step 5 — Impute temperature and rainfall columns
# Group statistic learned from TRAIN ONLY, applied to all splits (no leakage).
# Fallback chain: (Location, Month) -> Location -> global.

def fit_group_stats(train, col, agg):
    """Learn the imputation statistic at three granularities from the training set."""
    lm   = train.groupby(['Location', 'Month'])[col].agg(agg)  # per (Location, Month)
    loc  = train.groupby('Location')[col].agg(agg)             # per Location
    glob = train[col].agg(agg)                                 # global scalar
    return lm, loc, glob

def impute_col(frame, col, stats):
    """Fill NaNs in frame[col] using the fallback chain (Location, Month) -> Location -> global."""
    lm, loc, glob = stats
    idx = pd.MultiIndex.from_frame(frame[['Location', 'Month']])
    fill_lm  = pd.Series(lm.reindex(idx).to_numpy(), index=frame.index)
    fill_loc = frame['Location'].map(loc)
    return frame[col].fillna(fill_lm).fillna(fill_loc).fillna(glob)

# column -> aggregation: temperature uses mean, rainfall uses median
impute_plan = {
    'MinTemp': 'mean', 'MaxTemp': 'mean', 'Temp9am': 'mean', 'Temp3pm': 'mean',
    'Rainfall': 'median',
}

for col, agg in impute_plan.items():
    stats = fit_group_stats(df_train, col, agg)   # fit on train
    for part in (df_train, df_val, df_test):      # apply to all splits
        part[col] = impute_col(part, col, stats)

# Verify: no nulls remain in the imputed columns across any split
cols = list(impute_plan)
print("Remaining nulls after imputation:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", dict(part[cols].isna().sum()))

Remaining nulls after imputation:
  train {'MinTemp': np.int64(0), 'MaxTemp': np.int64(0), 'Temp9am': np.int64(0), 'Temp3pm': np.int64(0), 'Rainfall': np.int64(0)}
  val   {'MinTemp': np.int64(0), 'MaxTemp': np.int64(0), 'Temp9am': np.int64(0), 'Temp3pm': np.int64(0), 'Rainfall': np.int64(0)}
  test  {'MinTemp': np.int64(0), 'MaxTemp': np.int64(0), 'Temp9am': np.int64(0), 'Temp3pm': np.int64(0), 'Rainfall': np.int64(0)}


## Step 6 — Impute the high-missing columns (median) + missing-indicator flags

`Sunshine` (48%), `Evaporation` (43%), `Cloud3pm` (41%), `Cloud9am` (38%) are missing for large stretches (often whole stations/periods), so we treat them specially:

1. **Flag first** — add `<col>_was_missing` (1 where the original value was NaN, else 0), computed **before** imputing. This is a per-row check, so doing it on val/test is not leakage. The flag becomes a feature, letting the model use the *fact* a reading was missing.
2. **Then impute** the value with the **median** (these are skewed / bounded-okta columns), using the same train-fit fallback chain `(Location, Month)` → `Location` → global from Step 5.

Reuses `fit_group_stats` and `impute_col` defined in Step 5.

In [9]:
# Step 6 — Impute high-missing columns (median) + missing-indicator flags
high_missing = ['Sunshine', 'Evaporation', 'Cloud9am', 'Cloud3pm']

# 1) Flag BEFORE imputing — captures the original missingness pattern (per-row, no leakage)
for part in (df_train, df_val, df_test):
    for col in high_missing:
        part[col + '_was_missing'] = part[col].isna().astype(int)

# 2) Impute the value with the median; fit on TRAIN, apply to all splits
#    (reuses fit_group_stats / impute_col from Step 5; fallback (Loc,Month) -> Loc -> global)
for col in high_missing:
    stats = fit_group_stats(df_train, col, 'median')
    for part in (df_train, df_val, df_test):
        part[col] = impute_col(part, col, stats)

# Verify: no nulls remain, and show how much of each column was flagged as missing
flag_cols = [c + '_was_missing' for c in high_missing]
print("Remaining nulls after imputation:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", dict(part[high_missing].isna().sum()))

print("\nShare flagged as missing (train):")
print(df_train[flag_cols].mean().round(3).to_string())

Remaining nulls after imputation:
  train {'Sunshine': np.int64(0), 'Evaporation': np.int64(0), 'Cloud9am': np.int64(0), 'Cloud3pm': np.int64(0)}
  val   {'Sunshine': np.int64(0), 'Evaporation': np.int64(0), 'Cloud9am': np.int64(0), 'Cloud3pm': np.int64(0)}
  test  {'Sunshine': np.int64(0), 'Evaporation': np.int64(0), 'Cloud9am': np.int64(0), 'Cloud3pm': np.int64(0)}

Share flagged as missing (train):
Sunshine_was_missing       0.411
Evaporation_was_missing    0.375
Cloud9am_was_missing       0.361
Cloud3pm_was_missing       0.371


## Step 7 — Fill remaining `RainToday` nulls

`RainToday` describes the same event as the previous day's `RainTomorrow`, so we fill it with a cascade:

1. **Borrow `RainTomorrow[D−1]`** for the same Location when day D−1 is present and consecutive (a real label, verified to match 100% of the time). In this dataset the missing `RainToday` rows sit at date gaps, so this fills ~0 — but it's the most authentic source where it applies.
2. **Fallback — derive from `Rainfall`:** the dataset defines `RainToday = 1` iff `Rainfall > 1 mm` (verified to reproduce the label 100%). Since `Rainfall` was imputed in Step 5, this fills every remaining null.

In [10]:
# Step 7 — Fill remaining RainToday nulls
for part in (df_train, df_val, df_test):
    part.sort_values(['Location', 'Date'], inplace=True)
    prev_rt   = part.groupby('Location')['RainTomorrow'].shift(1)
    prev_date = part.groupby('Location')['Date'].shift(1)
    consecutive = (part['Date'] - prev_date).dt.days == 1
    # (a) authentic: previous-day RainTomorrow on consecutive days
    part['RainToday'] = part['RainToday'].fillna(prev_rt.where(consecutive))
    # (b) fallback: dataset definition RainToday = 1 iff Rainfall > 1mm (Rainfall imputed in Step 5)
    part['RainToday'] = part['RainToday'].fillna((part['Rainfall'] > 1).astype(int)).astype(int)

print("RainToday nulls after fill:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", int(part['RainToday'].isna().sum()))

RainToday nulls after fill:
  train 0
  val   0
  test  0


## Step 8 — Impute remaining numeric columns (Pressure, Humidity, WindSpeed, WindGustSpeed)

`Pressure9am/3pm`, `Humidity9am/3pm`, `WindGustSpeed`, `WindSpeed9am/3pm` are all low-missingness (≤10%), so no flags. Impute with the **median** via the same train-fit fallback chain `(Location, Month)` → `Location` → global (median is safe for the mildly-skewed wind/pressure/humidity distributions). Reuses `fit_group_stats` / `impute_col` from Step 5.

In [11]:
# Step 8 — Impute remaining numeric columns (median, no flags)
median_cols = ['Pressure9am', 'Pressure3pm', 'Humidity9am', 'Humidity3pm',
               'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm']

for col in median_cols:
    stats = fit_group_stats(df_train, col, 'median')   # fit on train
    for part in (df_train, df_val, df_test):           # apply to all splits
        part[col] = impute_col(part, col, stats)

print("Remaining nulls after Step 8:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", dict(part[median_cols].isna().sum()))

Remaining nulls after Step 8:
  train {'Pressure9am': np.int64(0), 'Pressure3pm': np.int64(0), 'Humidity9am': np.int64(0), 'Humidity3pm': np.int64(0), 'WindGustSpeed': np.int64(0), 'WindSpeed9am': np.int64(0), 'WindSpeed3pm': np.int64(0)}
  val   {'Pressure9am': np.int64(0), 'Pressure3pm': np.int64(0), 'Humidity9am': np.int64(0), 'Humidity3pm': np.int64(0), 'WindGustSpeed': np.int64(0), 'WindSpeed9am': np.int64(0), 'WindSpeed3pm': np.int64(0)}
  test  {'Pressure9am': np.int64(0), 'Pressure3pm': np.int64(0), 'Humidity9am': np.int64(0), 'Humidity3pm': np.int64(0), 'WindGustSpeed': np.int64(0), 'WindSpeed9am': np.int64(0), 'WindSpeed3pm': np.int64(0)}


## Step 9 — Fill wind-direction sin/cos nulls, drop raw direction strings

The raw `WindGustDir` / `WindDir9am` / `WindDir3pm` string columns are now redundant — their information lives in the `_sin`/`_cos` pairs from Step 3. We:

1. **Fill the `_sin`/`_cos` nulls with 0.** For a direction, `(sin, cos) = (0, 0)` is the zero-length wind vector — the natural neutral for "no/unknown prevailing direction", and it treats every missing direction identically instead of nudging it toward some median bearing.
2. **Drop the raw string columns** so only the numeric encodings remain.

After this the dataset is fully null-free.

In [12]:
# Step 9 — Fill wind-direction sin/cos nulls with 0, drop raw string columns
sincos_cols = [c + s for c in WIND_DIR_COLS for s in ('_sin', '_cos')]

for part in (df_train, df_val, df_test):
    part[sincos_cols] = part[sincos_cols].fillna(0)
    part.drop(columns=WIND_DIR_COLS, inplace=True)

# Final check: confirm the whole dataset is null-free across all splits
print("Total remaining nulls per split:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s} {int(part.isna().sum().sum())} nulls,  shape {part.shape}")

Total remaining nulls per split:
  train 0 nulls,  shape (98988, 32)
  val   0 nulls,  shape (17231, 32)
  test  0 nulls,  shape (25974, 32)


## Step 10 — Derived features

Add physically-motivated features that often predict rain better than the raw columns:

- **`PressureDrop` = `Pressure9am` − `Pressure3pm`** — falling pressure through the day signals an approaching front.
- **`HumidityChange` = `Humidity3pm` − `Humidity9am`** — rising afternoon humidity favours rain.
- **`TempRange` = `MaxTemp` − `MinTemp`** — a large diurnal range usually means clear, dry skies.
- **`Month_sin` / `Month_cos`** — cyclical encoding of the month so December and January are adjacent (seasonality matters enormously for rain).

These are deterministic row-wise transforms (no fitting), so applying them per split introduces no leakage.

In [13]:
# Step 10 — Derived features (deterministic row-wise transforms, no leakage)
for part in (df_train, df_val, df_test):
    part['PressureDrop']   = part['Pressure9am'] - part['Pressure3pm']
    part['HumidityChange'] = part['Humidity3pm'] - part['Humidity9am']
    part['TempRange']      = part['MaxTemp'] - part['MinTemp']
    part['Month_sin']      = np.sin(2 * np.pi * part['Month'] / 12)
    part['Month_cos']      = np.cos(2 * np.pi * part['Month'] / 12)

new_cols = ['PressureDrop', 'HumidityChange', 'TempRange', 'Month_sin', 'Month_cos']
print("Derived features added:", new_cols)
print(df_train[new_cols].describe().round(2).to_string())

Derived features added: ['PressureDrop', 'HumidityChange', 'TempRange', 'Month_sin', 'Month_cos']
       PressureDrop  HumidityChange  TempRange  Month_sin  Month_cos
count      98988.00        98988.00   98988.00   98988.00   98988.00
mean           2.38          -17.22      11.00      -0.04      -0.01
std            1.92           16.38       4.93       0.70       0.71
min          -25.70          -91.00      -1.69      -1.00      -1.00
25%            1.50          -28.00       7.20      -0.87      -0.87
50%            2.40          -17.00      10.40      -0.00      -0.00
75%            3.50           -6.00      14.40       0.50       0.87
max           25.80           91.00      31.20       1.00       1.00


## Step 11 — Assemble X / y, one-hot encode Location, scale features

Build the model matrices using a `ColumnTransformer` fit on **train only** (same fit/transform discipline as imputation):

- **Drop** `Date`, `Year`, `Month` (raw `Month` is redundant with `Month_sin/cos`), and the target.
- **Scale** the 19 continuous columns with `StandardScaler` (mean 0, sd 1) — required so logistic regression's coefficients are comparable and regularization is fair.
- **One-hot encode** `Location` with `drop_first` (48 dummies, `handle_unknown='ignore'` for safety).
- **Pass through** the already-bounded columns unchanged: wind & month sin/cos (in [−1, 1]), the 4 `_was_missing` flags, and `RainToday` (all 0/1 or [−1,1], so scaling them adds nothing).

`y = RainTomorrow`.

In [14]:
# Step 11 — Assemble X/y, one-hot encode Location, scale continuous features
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

target    = 'RainTomorrow'
drop_cols = ['Date', 'Year', 'Month', target]

scale_cols = ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed',
              'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am',
              'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm',
              'PressureDrop', 'HumidityChange', 'TempRange']
onehot_cols = ['Location']
# everything else (sin/cos, flags, RainToday) is passed through via remainder

def split_xy(part):
    return part.drop(columns=drop_cols), part[target].astype(int)

X_train_df, y_train = split_xy(df_train)
X_val_df,   y_val   = split_xy(df_val)
X_test_df,  y_test  = split_xy(df_test)

preprocess = ColumnTransformer(
    transformers=[
        ('scale',  StandardScaler(), scale_cols),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), onehot_cols),
    ],
    remainder='passthrough',          # bounded columns kept as-is
    verbose_feature_names_out=False,
)

X_train = preprocess.fit_transform(X_train_df)   # FIT on train only
X_val   = preprocess.transform(X_val_df)
X_test  = preprocess.transform(X_test_df)
feature_names = preprocess.get_feature_names_out()

print(f"X_train: {X_train.shape}   X_val: {X_val.shape}   X_test: {X_test.shape}")
print(f"# features: {len(feature_names)}")
print(f"y_train positive rate: {y_train.mean():.1%}   val: {y_val.mean():.1%}   test: {y_test.mean():.1%}")

X_train: (98988, 80)   X_val: (17231, 80)   X_test: (25974, 80)
# features: 80
y_train positive rate: 22.5%   val: 21.2%   test: 22.9%


## Step 12 — Logistic Regression baseline

A regularized linear baseline. **Fixed hyperparameters** (no search yet — this is the reference point future models beat):

| Param | Value | Reason |
|---|---|---|
| `penalty` | `l2` | Ridge; stable with the 80 (partly correlated) features |
| `C` | `1.0` | Default regularization strength — baseline |
| `solver` | `lbfgs` | Efficient for L2 on a dense 98,988 × 80 matrix |
| `max_iter` | `1000` | Raised from the default 100 to ensure convergence |
| `class_weight` | `balanced` | Re-weights classes to offset the 22.5% positive rate |
| `random_state` | `42` | Reproducibility |

**Protocol:** fit on **train**, report at the default 0.5 threshold, then pick the F1-optimal threshold on **validation** and apply that single fixed threshold to **test**. The test set is never used to choose anything. Because of imbalance we report precision / recall / F1 / ROC-AUC / PR-AUC, not just accuracy (the all-"No" baseline already scores ~77%).

In [15]:
# Step 12 — Logistic Regression baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             precision_recall_curve)

# --- Fixed hyperparameters (see markdown above) ---
logreg = LogisticRegression(
    C=1.0,                # L2 penalty is the default in sklearn 1.9 (explicit penalty='l2' is deprecated)
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
)
logreg.fit(X_train, y_train)
print(f"Converged in {logreg.n_iter_[0]} iterations\n")

# Predicted probabilities for the positive class (rain tomorrow)
p_val  = logreg.predict_proba(X_val)[:, 1]
p_test = logreg.predict_proba(X_test)[:, 1]

def report(y_true, p, thr, label):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yhat).ravel()
    print(f"{label}  (threshold={thr:.3f})")
    print(f"  accuracy {accuracy_score(y_true, yhat):.3f} | "
          f"precision {precision_score(y_true, yhat):.3f} | "
          f"recall {recall_score(y_true, yhat):.3f} | "
          f"F1 {f1_score(y_true, yhat):.3f}")
    print(f"  ROC-AUC {roc_auc_score(y_true, p):.3f} | PR-AUC {average_precision_score(y_true, p):.3f}")
    print(f"  confusion [TN {tn}  FP {fp} | FN {fn}  TP {tp}]\n")

# 1) Default 0.5 threshold
report(y_val,  p_val,  0.5, "VALIDATION @0.5")
report(y_test, p_test, 0.5, "TEST       @0.5")

# 2) Pick F1-optimal threshold on VALIDATION only, then freeze it for TEST
prec, rec, thr = precision_recall_curve(y_val, p_val)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = thr[f1s.argmax()]
print(f"--- F1-optimal threshold chosen on validation: {best_thr:.3f} ---\n")
report(y_val,  p_val,  best_thr, "VALIDATION @tuned")
report(y_test, p_test, best_thr, "TEST       @tuned")

Converged in 114 iterations

VALIDATION @0.5  (threshold=0.500)
  accuracy 0.809 | precision 0.536 | recall 0.738 | F1 0.621
  ROC-AUC 0.875 | PR-AUC 0.693
  confusion [TN 11251  FP 2332 | FN 957  TP 2691]

TEST       @0.5  (threshold=0.500)
  accuracy 0.788 | precision 0.525 | recall 0.766 | F1 0.623
  ROC-AUC 0.860 | PR-AUC 0.687
  confusion [TN 15911  FP 4117 | FN 1391  TP 4555]

--- F1-optimal threshold chosen on validation: 0.587 ---

VALIDATION @tuned  (threshold=0.587)
  accuracy 0.834 | precision 0.595 | recall 0.672 | F1 0.631
  ROC-AUC 0.875 | PR-AUC 0.693
  confusion [TN 11916  FP 1667 | FN 1198  TP 2450]

TEST       @tuned  (threshold=0.587)
  accuracy 0.814 | precision 0.579 | recall 0.694 | F1 0.631
  ROC-AUC 0.860 | PR-AUC 0.687
  confusion [TN 17024  FP 3004 | FN 1818  TP 4128]



In [16]:
# Step 12b — Inspect the linear equation inside the sigmoid:  z = b + sum_i w_i * x_i
coef = logreg.coef_[0]           # 80 weights, one per feature
b    = logreg.intercept_[0]      # bias term

coef_tbl = (pd.Series(coef, index=feature_names)
              .sort_values(key=np.abs, ascending=False))

print(f"Intercept (b): {b:+.4f}\n")
print("Top 15 features by |weight|  (sign = direction of effect on log-odds of rain):")
print(coef_tbl.head(15).round(4).to_string())

# Reconstruct the equation for one example row (test row 0) to show z explicitly
x0 = X_test[0]
z0 = b + np.dot(coef, x0)
print(f"\nExample (test row 0):  z = {z0:+.3f}  ->  sigma(z) = P(rain) = {1/(1+np.exp(-z0)):.3f}")
print(f"sklearn predict_proba check: {logreg.predict_proba(X_test[:1])[0,1]:.3f}")

Intercept (b): -0.2739

Top 15 features by |weight|  (sign = direction of effect on log-odds of rain):
Location_Katherine          -1.6968
Location_Townsville         -1.6139
Location_Wollongong         -1.5882
Location_Darwin             -1.3922
Location_NorahHead          -1.1682
Location_MountGinini        -1.1586
Location_GoldCoast          -1.0281
Location_NorfolkIsland      -0.9208
Location_Cairns             -0.8823
Location_MelbourneAirport   -0.8811
Location_Ballarat           -0.8515
Location_Hobart             -0.8047
Location_Launceston         -0.7922
Location_Newcastle          -0.7827
Humidity3pm                  0.7692

Example (test row 0):  z = -2.689  ->  sigma(z) = P(rain) = 0.064
sklearn predict_proba check: 0.064


In [17]:
# Step 12c — Loss on train vs validation: underfit / overfit / normal?
from sklearn.metrics import log_loss
from sklearn.utils.class_weight import compute_sample_weight

p_train = logreg.predict_proba(X_train)[:, 1]

# Class weights ('balanced') as used inside the training objective
sw_train = compute_sample_weight('balanced', y_train)
sw_val   = compute_sample_weight('balanced', y_val)   # uses val's own class counts

# 1) Mean (per-sample) binary cross-entropy — the apples-to-apples comparison
ll_tr  = log_loss(y_train, p_train)
ll_val = log_loss(y_val,   p_val)

# 2) Class-weighted mean BCE (matches what the optimizer actually minimized)
llw_tr  = log_loss(y_train, p_train, sample_weight=sw_train)
llw_val = log_loss(y_val,   p_val,   sample_weight=sw_val)

# 3) Full regularized objective on TRAIN:  J = 0.5||w||^2 + C * sum_i c_yi * BCE_i   (C=1)
C = 1.0
penalty = 0.5 * np.dot(coef, coef)
weighted_bce_sum = log_loss(y_train, p_train, sample_weight=sw_train, normalize=False)
J_train = penalty + C * weighted_bce_sum

print("Per-sample log-loss (unweighted):")
print(f"  train {ll_tr:.4f}   |   val {ll_val:.4f}   |   gap {ll_val-ll_tr:+.4f}")
print("\nPer-sample log-loss (class-weighted, matches training objective):")
print(f"  train {llw_tr:.4f}   |   val {llw_val:.4f}   |   gap {llw_val-llw_tr:+.4f}")
print("\nFull regularized objective on train:")
print(f"  J = 0.5||w||^2 + C*weighted_BCE_sum = {penalty:.1f} + {weighted_bce_sum:.1f} = {J_train:.1f}")
print(f"  (||w||^2 = {np.dot(coef,coef):.2f},  n_train = {len(y_train):,})")

# Reference points for context
from sklearn.metrics import log_loss as _ll
base_rate = y_train.mean()
naive = _ll(y_val, np.full_like(p_val, base_rate))
print(f"\nReference: a constant-{base_rate:.3f} predictor scores log-loss {naive:.4f} on val")

Per-sample log-loss (unweighted):
  train 0.4426   |   val 0.4167   |   gap -0.0258

Per-sample log-loss (class-weighted, matches training objective):
  train 0.4431   |   val 0.4439   |   gap +0.0009

Full regularized objective on train:
  J = 0.5||w||^2 + C*weighted_BCE_sum = 13.8 + 43859.2 = 43873.0
  (||w||^2 = 27.56,  n_train = 98,988)

Reference: a constant-0.225 predictor scores log-loss 0.5167 on val


In [18]:
# Step 12d — Does changing C (regularization) move the bias? Sweep C and watch the loss.
# Smaller C = stronger reg (more bias);  larger C = weaker reg (approaches unregularized linear fit).
print(f"{'C':>10} {'||w||^2':>9} {'train_ll':>9} {'val_ll':>9} {'val_AUC':>8}")
for Cval in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 10000.0]:
    m = LogisticRegression(C=Cval, solver='lbfgs', max_iter=2000,
                           class_weight='balanced', random_state=42).fit(X_train, y_train)
    ptr = m.predict_proba(X_train)[:, 1]
    pv  = m.predict_proba(X_val)[:, 1]
    print(f"{Cval:>10} {np.dot(m.coef_[0], m.coef_[0]):>9.2f} "
          f"{log_loss(y_train, ptr):>9.4f} {log_loss(y_val, pv):>9.4f} "
          f"{roc_auc_score(y_val, pv):>8.3f}")
print("\nTrain loss barely moves once C >= ~1: the L2 penalty is no longer the constraint —")
print("the linear hypothesis class is. C cannot reduce model-class bias.")

         C   ||w||^2  train_ll    val_ll  val_AUC


     0.001      1.83    0.4503    0.4263    0.870


      0.01      5.54    0.4441    0.4182    0.874


       0.1     11.25    0.4429    0.4167    0.875


       1.0     27.56    0.4426    0.4167    0.875


      10.0     35.47    0.4425    0.4168    0.875


     100.0     37.07    0.4425    0.4171    0.875


   10000.0     36.89    0.4425    0.4170    0.875

Train loss barely moves once C >= ~1: the L2 penalty is no longer the constraint —
the linear hypothesis class is. C cannot reduce model-class bias.


## Step 13 — Logistic Regression with polynomial (interaction) features

The diagnosis in Step 12c/d was **high model-class bias, not overfitting**: train loss sat on a 0.4425 floor that regularization couldn't move, because a linear boundary is the ceiling. This step adds capacity *inside the LR family* by expanding the feature space so the still-linear model can represent curvature and interactions.

**Design choices:**
- **Degree-2 `PolynomialFeatures` (squares + pairwise interactions) on the 19 continuous columns only.** Squaring 0/1 flags or interacting 47 one-hot `Location` dummies adds mostly-zero, meaningless columns — so dummies, sin/cos, flags and `RainToday` are **passed through untouched**. This keeps the matrix at ~270 features instead of ~3,300.
- **Scale *after* expansion** (`StandardScaler` on the polynomial block) so L2 regularizes every term fairly.
- **Re-tune `C`.** Capacity now matters: with hundreds of interaction terms, overfitting becomes possible, so `C` will actually move validation loss (unlike the linear model). We pick `C` by **validation log-loss**, fit on train only.

**Success check:** does train loss drop below the 0.4425 linear floor (bias reduced) *and* does validation loss / AUC improve (the gain generalizes)?

In [19]:
# Step 13 — Logistic Regression with degree-2 polynomial features (continuous cols only)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

# Expand continuous cols -> squares + pairwise interactions, then standardize the expansion.
cont_poly = Pipeline([
    ('poly',  PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)),
    ('scale', StandardScaler()),
])

preprocess_poly = ColumnTransformer(
    transformers=[
        ('contpoly', cont_poly, scale_cols),                                   # 19 -> 209
        ('onehot',   OneHotEncoder(drop='first', handle_unknown='ignore',
                                   sparse_output=False), onehot_cols),         # Location dummies
    ],
    remainder='passthrough',          # sin/cos, _was_missing flags, RainToday: untouched
    verbose_feature_names_out=False,
)

Xp_train = preprocess_poly.fit_transform(X_train_df)   # FIT on train only
Xp_val   = preprocess_poly.transform(X_val_df)
Xp_test  = preprocess_poly.transform(X_test_df)
print(f"Feature count: {X_train.shape[1]} (linear)  ->  {Xp_train.shape[1]} (polynomial)\n")

# Re-tune C on validation log-loss (capacity now matters -> regularization is live again)
LINEAR_FLOOR = 0.4425   # train log-loss the linear model could not beat (Step 12d)
print(f"{'C':>8} {'train_ll':>9} {'val_ll':>9} {'val_AUC':>8}")
best = None
for Cval in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:
    m = LogisticRegression(C=Cval, solver='lbfgs', max_iter=3000,
                           class_weight='balanced', random_state=42).fit(Xp_train, y_train)
    ll_tr = log_loss(y_train, m.predict_proba(Xp_train)[:, 1])
    pv    = m.predict_proba(Xp_val)[:, 1]
    ll_v  = log_loss(y_val, pv)
    auc_v = roc_auc_score(y_val, pv)
    print(f"{Cval:>8} {ll_tr:>9.4f} {ll_v:>9.4f} {auc_v:>8.3f}")
    if best is None or ll_v < best[1]:
        best = (Cval, ll_v, m)

best_C, _, poly_model = best
print(f"\nBest C by validation log-loss: {best_C}")

# Refit summary at best C
p_tr = poly_model.predict_proba(Xp_train)[:, 1]
p_v  = poly_model.predict_proba(Xp_val)[:, 1]
p_te = poly_model.predict_proba(Xp_test)[:, 1]
print(f"train log-loss {log_loss(y_train, p_tr):.4f}  (linear floor was {LINEAR_FLOOR})  "
      f"-> {'BIAS REDUCED' if log_loss(y_train, p_tr) < LINEAR_FLOOR else 'no improvement'}")
print(f"val   log-loss {log_loss(y_val, p_v):.4f}  |  train-val gap {log_loss(y_val,p_v)-log_loss(y_train,p_tr):+.4f}\n")

# Threshold tuned on validation (F1), frozen for test  -- reuse report() from Step 12
prec, rec, thr = precision_recall_curve(y_val, p_v)
f1s = 2*prec[:-1]*rec[:-1] / (prec[:-1]+rec[:-1]+1e-12)
poly_thr = thr[f1s.argmax()]
print(f"F1-optimal threshold on validation: {poly_thr:.3f}\n")
report(y_test, p_te, poly_thr, "TEST  poly @tuned")

print("Linear baseline for comparison (Step 12): TEST ROC-AUC 0.860 | PR-AUC 0.687 | F1 0.631")
print(f"Polynomial model:                        TEST ROC-AUC {roc_auc_score(y_test,p_te):.3f} | "
      f"PR-AUC {average_precision_score(y_test,p_te):.3f}")

Feature count: 80 (linear)  ->  270 (polynomial)

       C  train_ll    val_ll  val_AUC


   0.001    0.4394    0.4149    0.877


   0.003    0.4351    0.4102    0.879


    0.01    0.4323    0.4069    0.881


    0.03    0.4311    0.4055    0.881


     0.1    0.4303    0.4051    0.881


     0.3    0.4300    0.4053    0.882


     1.0    0.4295    0.4054    0.882


     3.0    0.4290    0.4051    0.882

Best C by validation log-loss: 0.1
train log-loss 0.4303  (linear floor was 0.4425)  -> BIAS REDUCED
val   log-loss 0.4051  |  train-val gap -0.0251

F1-optimal threshold on validation: 0.581

TEST  poly @tuned  (threshold=0.581)
  accuracy 0.820 | precision 0.591 | recall 0.700 | F1 0.641
  ROC-AUC 0.865 | PR-AUC 0.703
  confusion [TN 17146  FP 2882 | FN 1782  TP 4164]

Linear baseline for comparison (Step 12): TEST ROC-AUC 0.860 | PR-AUC 0.687 | F1 0.631
Polynomial model:                        TEST ROC-AUC 0.865 | PR-AUC 0.703
